<a href="https://colab.research.google.com/github/AbhishekChaganti/MY_REPO/blob/main/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================= INSTALL =================
!pip install -q datasets pillow scikit-learn

# ================= IMPORTS =================
import numpy as np
from datasets import load_dataset
from PIL import Image
from sklearn.model_selection import train_test_split
import os

# ================= LOAD DATASET =================
print("=== Downloading Dataset ===")
dataset = load_dataset("JamieWithofs/Deepfake-and-real-images", split="train")

print("Total samples in dataset:", len(dataset))

# ================= BALANCE DATA =================
real_images = []
fake_images = []

MAX_PER_CLASS = 5000

for item in dataset:
    label = item['label']  # 0 = real, 1 = fake

    img = item['image'].convert('RGB').resize((224, 224))
    img_array = np.array(img) / 255.0

    if label == 0 and len(real_images) < MAX_PER_CLASS:
        real_images.append(img_array)

    elif label == 1 and len(fake_images) < MAX_PER_CLASS:
        fake_images.append(img_array)

    # Stop when both reached
    if len(real_images) >= MAX_PER_CLASS and len(fake_images) >= MAX_PER_CLASS:
        break

print(f"Collected REAL: {len(real_images)}, FAKE: {len(fake_images)}")

# ================= COMBINE =================
images = np.array(real_images + fake_images)
labels = np.array([0]*len(real_images) + [1]*len(fake_images))

print("Final dataset shape:", images.shape)
print("Labels distribution:", np.bincount(labels))

# ================= SHUFFLE =================
indices = np.random.permutation(len(images))
images = images[indices]
labels = labels[indices]

# ================= TRAIN TEST SPLIT =================
train_images, test_images, train_labels, test_labels = train_test_split(
    images, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print(f"Train: {train_images.shape}, Test: {test_images.shape}")

# ================= SAVE FILES =================
save_dir = "/content/models"
os.makedirs(save_dir, exist_ok=True)

np.save(os.path.join(save_dir, "train_images.npy"), train_images)
np.save(os.path.join(save_dir, "train_labels.npy"), train_labels)
np.save(os.path.join(save_dir, "test_images.npy"), test_images)
np.save(os.path.join(save_dir, "test_labels.npy"), test_labels)

print("✅ Dataset saved in /content/models/")

In [ ]:
# ================= INSTALL =================
!pip install -q transformers matplotlib seaborn

# ================= IMPORTS =================
import numpy as np
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import os
from google.colab import files

# ================= LOAD TEST DATA =================
models_dir = "/content/models"

test_images = np.load(os.path.join(models_dir, 'test_images.npy'))
test_labels = np.load(os.path.join(models_dir, 'test_labels.npy'))

print("Test shape:", test_images.shape)
print("Label distribution:", np.bincount(test_labels))

# ================= LOAD MODEL =================
model_name = "dima806/deepfake_vs_real_image_detection"

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# ================= BATCH PREDICTION =================
preds = []
batch_size = 32

for i in range(0, len(test_images), batch_size):
    batch = test_images[i:i+batch_size]

    inputs = processor(batch, return_tensors="pt", do_rescale=False).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        batch_preds = outputs.logits.argmax(-1).cpu().numpy()
        preds.extend(batch_preds)

preds = np.array(preds)

# ⚠️ Adjust if needed
preds = 1 - preds

# ================= METRICS =================
acc = accuracy_score(test_labels, preds)
cm = confusion_matrix(test_labels, preds)

print(f"\n🎯 Accuracy: {acc*100:.2f}%")
print("Confusion Matrix:\n", cm)

# ================= SAVE GRAPH =================
save_path = "/content/confusion_matrix.png"

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["Fake", "Real"],
            yticklabels=["Fake", "Real"])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix\nAccuracy: {acc*100:.2f}%")

plt.tight_layout()
plt.savefig(save_path)   # ✅ SAVE PNG
plt.show()

print(f"\n✅ Saved to: {save_path}")

# ================= DOWNLOAD =================
from google.colab import files
files.download('/content/confusion_matrix.png')